# Classificação de Dengue — KNN x SVM

Disciplina de Inteligência Artificial — Professor Munif — Unicesumar 2026

Este notebook treina e compara dois modelos de classificação para prever dengue
a partir de dados clínicos/hematológicos:
- **Parte 1:** KNN (k-Nearest Neighbors)
- **Parte 2:** SVM (Support Vector Machine)

Ao final, os modelos treinados são salvos em `.pkl` para uso no back-end do app.


## 1. Imports e setup

In [ ]:
# Instala o imbalanced-learn caso nao esteja disponivel (ex.: Google Colab)
try:
    import imblearn
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc)
import joblib

# Pipeline do imbalanced-learn (suporta SMOTE dentro do pipeline)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE


## 2. Carregamento dos dados

No **Google Colab**, faça o upload do arquivo CSV antes de rodar esta célula
(use o painel de arquivos à esquerda ou `files.upload()`).

> Observação: o tratamento de outliers foi propositalmente **não aplicado** para não
> descartar pacientes com quadros severos de trombocitopenia (queda brusca de plaquetas),
> que são justamente casos clinicamente relevantes para dengue.

In [ ]:
# Caminho do arquivo (ajuste se necessário no Colab)
CSV_PATH = 'Dengue_diseases_dataset_modified (1).csv'

df = pd.read_csv(CSV_PATH)
print('Formato:', df.shape)
df.head()


## 3. Análise exploratória rápida (EDA)

In [ ]:
print(df.info())
print('\nDistribuicao da classe alvo (dengue_label):')
print(df['dengue_label'].value_counts())

df['dengue_label'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'])
plt.title('Distribuicao das classes (0 = sem dengue, 1 = dengue)')
plt.xlabel('Classe'); plt.ylabel('Quantidade'); plt.xticks(rotation=0)
plt.show()


## 4. Separação de features/target e divisão treino/teste

Divisão 80/20 **estratificada** para manter a proporção das classes.

In [ ]:
X = df.drop('dengue_label', axis=1)
y = df['dengue_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print('Treino:', X_train.shape, ' Teste:', X_test.shape)


## 5. Pré-processamento

- **Numéricas:** imputação pela mediana + `StandardScaler` (escalonamento obrigatório para KNN e SVM).
- **Categórica (`gender`):** imputação pela moda + `OneHotEncoder` (Male/Female/Child).

In [ ]:
numeric_features = ['age', 'hemoglobin_g_dl', 'wbc_count', 'differential_count',
                    'rbc_count', 'platelet_count', 'platelet_distribution_width']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['gender']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


## 6. Definição dos modelos (Pipeline = pré-processamento + algoritmo)

Encapsular o pré-processamento dentro do pipeline é importante: o `.pkl` salvo já
sabe transformar dados crus vindos do formulário, sem precisar repetir o
pré-processamento no back-end.

In [ ]:
knn_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

svm_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    # probability=True e necessario para plotar a Curva ROC do SVM
    ('classifier', SVC(kernel='rbf', C=1.0, probability=True, random_state=42))
])


## 7. Treinamento

In [ ]:
print('Treinando modelos (mantendo todos os casos extremos do dataset)...')
knn_pipeline.fit(X_train, y_train)
svm_pipeline.fit(X_train, y_train)
print('Treinamento concluido.')


## 8. Avaliação — métricas e matrizes de confusão

In [ ]:
modelos = {'KNN': knn_pipeline, 'SVM': svm_pipeline}
resultados = {}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (nome, modelo) in enumerate(modelos.items()):
    y_pred = modelo.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    resultados[nome] = [acc, prec, rec, f1]

    print(f'\n--- Relatorio de Classificacao: {nome} ---')
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo.classes_)
    disp.plot(ax=axes[idx], cmap='Blues')
    axes[idx].set_title(f'Matriz de Confusao - {nome}')

plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=120, bbox_inches='tight')
plt.show()


## 9. Curva ROC / AUC

In [ ]:
plt.figure(figsize=(8, 6))
for nome, modelo in modelos.items():
    y_prob = modelo.predict_proba(X_test)[:, 1]  # probabilidade da classe 1 (Dengue)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{nome} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curva ROC - Comparacao KNN x SVM')
plt.legend(loc='lower right')
plt.savefig('curva_roc.png', dpi=120, bbox_inches='tight')
plt.show()


## 10. Comparação gráfica entre os modelos

In [ ]:
df_resultados = pd.DataFrame(resultados, index=['Acuracia', 'Precisao', 'Revocacao', 'F1-Score'])
print(df_resultados)

df_resultados.plot(kind='bar', figsize=(10, 6), colormap='Set2')
plt.title('Comparacao de Metricas entre KNN e SVM')
plt.ylabel('Score'); plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.savefig('comparacao_metricas.png', dpi=120, bbox_inches='tight')
plt.show()


## 11. Salvando os modelos treinados

Os arquivos `.pkl` serão usados pelo back-end do app para prever a partir do formulário.

In [ ]:
joblib.dump(knn_pipeline, 'modelo_knn.pkl')
joblib.dump(svm_pipeline, 'modelo_svm.pkl')
print("Modelos salvos: 'modelo_knn.pkl' e 'modelo_svm.pkl'")


## 12. Teste rápido de predição (simulando o formulário)

Exemplo de como o back-end usará o modelo: recebe os dados crus do formulário e prevê.

In [ ]:
exemplo = pd.DataFrame([{
    'age': 35,
    'gender': 'Male',
    'hemoglobin_g_dl': 13.5,
    'wbc_count': 3000,
    'differential_count': 1,
    'rbc_count': 1,
    'platelet_count': 45000,
    'platelet_distribution_width': 17.0
}])

pred = svm_pipeline.predict(exemplo)[0]
prob = svm_pipeline.predict_proba(exemplo)[0][1]
print(f'Predicao SVM: {pred}  (0=sem dengue, 1=dengue)')
print(f'Probabilidade de dengue: {prob:.2%}')
